---
Hardware
description: This lesson explores modern quantum computing hardware. 
---

# Hardware

## 1. Introduction

This lesson explores modern quantum computing hardware.

We will start by verifying some versions and importing some relevant packages.



In [2]:
import sys

recommended = (3, 10)
current = sys.version_info[:2]

if current < recommended:
    print(f"You are using Python {current[0]}.{current[1]}. Python {recommended[0]}.{recommended[1]} or later is recommended.")
else:
    print(f"Python {current[0]}.{current[1]} — looks good!")

Python 3.13 — looks good!


In [3]:
import os
import platform
import shutil
import subprocess
import sys

import qiskit

from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import array_to_latex, plot_histogram, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt




ModuleNotFoundError: No module named 'qiskit'

## 2. Backend and Target

Qiskit provides an API to obtain the information, both static and dynamic, about a quantum device. We use a Backend instance to interface with a device, which includes a Target instance, an abstract machine model that summarizes the pertinent features such as its instruction set architecture (ISA) and any properties or constraints associated with it.
Let us use these backend instances to get some of the information you see on the [Compute resources](/computers) page
on IBM Quantum® Platform.   First, we create a backend instance for a device of interest.  In the following, we pick "ibm\_kyoto" , "ibm\_kawasaki" or the least busy Eagle machine. Your access to QPUs might differ; update the backend name accordingly.



In [1]:
# Save your API key to track your progress and have access to the quantum computers

your_api_key = "6bGNi4s4dZrYB_JYrGKdocbZCKQhjcKugqeWKsHM0iCS"

your_crn = "crn:v1:bluemix:public:quantum-computing:us-east:a/4171228ed9e64af9b7cb12da8fe25466:eb4e3526-e437-4cf5-9b55-3dc76f65adf3::"


In [6]:
service = QiskitRuntimeService()

In [4]:
service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
) 
backend.name

'ibm_fez'

We start with some basic (static) information about the device.



In [5]:
print(
    f"""
{backend.name}, {backend.num_qubits} qubits
processor type = {backend.processor_type}
basis gates = {backend.basis_gates}
"""
)


ibm_fez, 156 qubits
processor type = {'family': 'Heron', 'revision': '2'}
basis gates = ['cz', 'id', 'rz', 'sx', 'x']



### 2.1 Exercise

Try to get the basic information about a Heron device, "ibm\_fez". Try this on your own, but code has been added below for you to check yourself.



In [ ]:
backend = service.backend("ibm_fez")  # a Heron

# your code here
print(
    f"""
{backend.name}, {backend.num_qubits} qubits
processor type = {backend.processor_type}
basis gates = {backend.basis_gates}
"""
)

### 2.2 Coupling map

We now draw the coupling map of the device. As you can see, nodes are qubits which are numbered. Edges indicate pairs to which you can directly apply the 2-qubit entangling gate.  The topology is called a "heavy-hex lattice".



In [ ]:
# This function requires that Graphviz is installed. If you need to install Graphviz you can refer to:
# https://graphviz.org/download/#executable-packages for instructions.
try:
    fig = backend.coupling_map.draw()
except RuntimeError as ex:
    print(ex)
fig

## 3. Qubit properties

The ibm_fez device has 156 qubits.   Let us obtain the properties of some of them.



In [ ]:
for qn in range(backend.num_qubits):
    if qn >= 5:
        break
    print(f"{qn}: {backend.qubit_properties(qn)}")

In [ ]:
import statistics

Let us calculate the median of T1 times of the qubits.   Compare the result to the one shown for the device on [IBM Quantum Platform.](/)



In [ ]:
t1s = [backend.qubit_properties(qq).t1 for qq in range(backend.num_qubits)]
f"Median T1: {(statistics.median(t1s)*10**6):.2f} \u03bcs"

In [ ]:
target = backend.target
target.keys()

Its values are also dictionaries.  Let us look at some of the items of the value (dictionary) for the 'sx' operation.



### 3.2 Gate and readout errors

We now turn to gate errors. To begin with, we study the data structure of the target instance. It is a dictionary whose keys are operation names.



In [ ]:
for i, qq in enumerate(target["sx"]):
    if i >= 5:
        break
    print(i, qq, target["sx"][qq])

Let us do the same for the 'rz' and 'measure' operations.



In [ ]:
for i, edge in enumerate(target["rz"]):
    if i >= 5:
        break
    print(i, edge, target["rz"][edge])

In [ ]:
for i, qq in enumerate(target["measure"]):
    if i >= 5:
        break
    print(i, qq, target["measure"][qq])

As you can see, the errors of readout tend to be larger than those of the 2-qubit operation, which in turn tend to be larger than the 1-qubit operation.

Having understood the data structures, we are ready to calculate the median errors for the 'rz' and the 'sx' gates. Again, compare the results with the ones shown for the device on the [IBM Quantum Platform.](/)



In [ ]:
sx_errors = [inst_prop.error for inst_prop in target["sx"].values()]
f"Median SX error: {(statistics.median(sx_errors)):.3e}"

In [ ]:
rz_errors = [inst_prop.error for inst_prop in target["rz"].values()]
f"Median rz error: {(statistics.median(rz_errors)):.3e}"

## 4. Appendix



A popular feature of Qiskit is its visualization capability. It includes circuit visualizers, state and distribution visualizers, and target visualizer. Let us use some capabilities of the target visualizer.



In [ ]:
from qiskit.visualization import plot_gate_map

plot_gate_map(backend, font_size=14)

In [ ]:
from qiskit.visualization import plot_error_map

plot_error_map(backend)

In [ ]:
# Check Qiskit version
import qiskit

qiskit.__version__